# Clase 120 — Funciones y grafos (AutoGraph)

`@tf.function` compila una función Python a un **grafo TF** estático: acelera
2-10× y permite deploy (TF Serving / TFLite). **AutoGraph** traduce `if`/`for`/
`while` de Python a `tf.cond`/`tf.while_loop` automáticamente.

Requiere: `tensorflow` (≥ 2.x), `time`.

## 1. Eager vs graph: `@tf.function`

In [ ]:
import time
import tensorflow as tf
tf.random.set_seed(42)

def f_eager(x):
    return tf.reduce_sum(x ** 2 + 3 * x + 1)

@tf.function
def f_grafo(x):
    return tf.reduce_sum(x ** 2 + 3 * x + 1)

x = tf.random.normal((1000,))
print("eager:", float(f_eager(x)))
print("grafo:", float(f_grafo(x)))
print("tipo de f_grafo:", type(f_grafo).__name__)

## 2. Medir el speedup en un loop

In [ ]:
x = tf.random.normal((1000, 1000))

t0 = time.perf_counter()
for _ in range(100):
    f_eager(x)
t_eager = time.perf_counter() - t0

_ = f_grafo(x)                       # 1ª llamada: traza el grafo (no se cronometra)
t0 = time.perf_counter()
for _ in range(100):
    f_grafo(x)
t_grafo = time.perf_counter() - t0

print(f"eager: {t_eager:.4f}s | grafo: {t_grafo:.4f}s")
print(f"speedup: {t_eager / max(t_grafo, 1e-9):.2f}x")

## 3. AutoGraph: `if`/`for` → `tf.cond`/`tf.while_loop`

In [ ]:
@tf.function
def suma_positivos(x):
    total = tf.constant(0.0)
    for v in x:                      # AutoGraph -> tf.while_loop
        if v > 0:                    # AutoGraph -> tf.cond
            total += v
    return total

muestra = tf.constant([1.0, -2.0, 3.0, -4.0, 5.0])
print("suma de positivos:", float(suma_positivos(muestra)))

# ver el Python que generó AutoGraph:
codigo = tf.autograph.to_code(suma_positivos.python_function)
print(codigo[:400])

## 4. `print` (tracing) vs `tf.print` (runtime)

In [ ]:
@tf.function
def con_prints(x):
    print("PYTHON print: solo se ejecuta al TRAZAR")
    tf.print("TF print: en CADA ejecución, x =", x)
    return x * 2

print("--- 1ra llamada (traza + ejecuta) ---")
con_prints(tf.constant(1.0))
print("--- 2da llamada (solo ejecuta) ---")
con_prints(tf.constant(2.0))

## 5. Retracing e `input_signature`

In [ ]:
@tf.function
def sin_firma(x):
    tf.print("  trazando shape", tf.shape(x))
    return tf.reduce_sum(x)

print("Sin input_signature (retrace por shape distinta):")
sin_firma(tf.random.normal((2,)))
sin_firma(tf.random.normal((3,)))    # shape nueva -> retrace
sin_firma(tf.random.normal((2,)))    # shape ya vista -> reutiliza

@tf.function(input_signature=[tf.TensorSpec(shape=[None, 784], dtype=tf.float32)])
def con_firma(x):
    return tf.reduce_mean(x, axis=1)

print("Con input_signature (dim batch None, sin retrace):")
print(con_firma(tf.random.normal((4, 784))).shape)
print(con_firma(tf.random.normal((8, 784))).shape)

## 6. `tf.TensorArray`: reemplaza `list.append` dentro del grafo

In [ ]:
@tf.function
def acumular(x):
    ta = tf.TensorArray(tf.float32, size=tf.shape(x)[0])
    for i in tf.range(tf.shape(x)[0]):
        ta = ta.write(i, x[i] * 2.0)
    return ta.stack()

print("TensorArray (append no funciona en grafo):",
      acumular(tf.constant([1.0, 2.0, 3.0])).numpy())

## Ejercicios

1. **Speedup básico**: definí `f(x) = tf.reduce_sum(x**2 + 3*x + 1)` y medí el
   tiempo eager vs `@tf.function` en un loop de miles de iteraciones.
2. **Retracing**: llamá una `@tf.function` con tensores de shapes distintas y
   detectá los retraces con un `tf.print` de `tf.shape(x)`.
3. **AutoGraph**: escribí una función con `for` + `if` y verificá su traducción
   con `tf.autograph.to_code(f.python_function)`.
4. **`tf.print` vs `print`**: demostrá que `print` solo corre en la 1ra llamada
   (tracing) y `tf.print` en todas.
5. **`input_signature`**: fijá una firma `(None, 784)` para evitar retracing al
   cambiar el tamaño de batch.

## Conclusiones

- `@tf.function` convierte Python en un **grafo** reutilizable: más rápido y deployable.
- **AutoGraph** traduce control de flujo Python (`if`/`for`/`while`) a ops TF (`tf.cond`/`tf.while_loop`).
- El **retracing** ocurre al cambiar shape/dtype de los inputs; `input_signature` con `TensorSpec` lo evita.
- `print` corre solo en tracing; para logs en ejecución se usa `tf.print`.
- Las listas Python no viven en el grafo: se usa `tf.TensorArray`. Debuggear en eager, luego decorar.

## ✅ Soluciones de los ejercicios

`tf.function`, grafos y AutoGraph (cap. 12). Se validan por AST sin TF. Cubren el speedup eager->graph, el retracing por shape, la conversión AutoGraph de `for`/`if`, `tf.print` vs `print` e `input_signature` para evitar retraces.

**Ej. 1 — Speedup básico.** Comparar tiempo eager vs `tf.function` en 10 000 iteraciones.

In [ ]:
import time
import tensorflow as tf

def f(x):
    return tf.reduce_sum(x ** 2 + 3 * x + 1)

f_graph = tf.function(f)
x = tf.random.normal((1000,))
f_graph(x)   # warmup: traza el grafo la 1a vez

t0 = time.perf_counter(); [f(x) for _ in range(10_000)];       eager = time.perf_counter() - t0
t0 = time.perf_counter(); [f_graph(x) for _ in range(10_000)]; graph = time.perf_counter() - t0
print(f"eager {eager:.3f}s | graph {graph:.3f}s | speedup x{eager / graph:.2f}")

**Ej. 2 — Retracing.** Cada shape nueva fuerza un *retrace*; se detecta con un `print` dentro de la función.

In [ ]:
import tensorflow as tf

@tf.function
def g(x):
    print("TRACING con shape", x.shape)   # solo se ejecuta al TRAZAR (no en cada call)
    return tf.reduce_sum(x)

g(tf.zeros((2,)))   # traza (shape (2,))
g(tf.zeros((3,)))   # RE-traza (shape distinta)
g(tf.zeros((2,)))   # reutiliza el grafo -> no imprime
print("cada shape nueva crea una ConcreteFunction (retrace costoso)")

**Ej. 3 — AutoGraph.** Un `for`/`if` de Python se convierte a ops de grafo; se inspecciona con `to_code`.

In [ ]:
import tensorflow as tf

@tf.function
def h(x):
    s = 0.0
    for i in tf.range(10):          # for sobre tensor -> tf.while_loop
        if i % 2 == 0:              # if sobre tensor -> tf.cond
            s += tf.cast(i, tf.float32)
    return s

print(tf.autograph.to_code(h.python_function))   # muestra el codigo generado por AutoGraph

**Ej. 4 — `tf.print` vs `print`.** `print` solo corre en tracing; `tf.print` corre en cada ejecución.

In [ ]:
import tensorflow as tf

@tf.function
def k(x):
    print("python print: solo en tracing")   # 1 sola vez (al trazar)
    tf.print("tf.print: en cada ejecucion")  # es una op del grafo -> siempre
    return x + 1

k(tf.constant(1.0))   # traza + ejecuta -> salen ambos
k(tf.constant(2.0))   # solo ejecuta -> sale tf.print, NO print

**Ej. 5 — `input_signature`.** Fijar `(None, 784)` para no retrazar al cambiar el batch size.

In [ ]:
import tensorflow as tf

@tf.function(input_signature=[tf.TensorSpec(shape=(None, 784), dtype=tf.float32)])
def forward(x):
    return tf.reduce_mean(x, axis=1)

forward(tf.zeros((32, 784)))   # batch 32
forward(tf.zeros((64, 784)))   # batch 64 -> NO retraza (dim 0 = None)
print("input_signature (None, 784): un unico grafo para cualquier batch size")